
# Notebook 18 · GenAI para personalización de copy y creatividad por microsegmento

## Pregunta de negocio
¿Cómo usar desempeño histórico, segmentación y guardrails para preparar copy personalizado con IA generativa?

## Qué aprenderás
- Cómo elegir señales útiles para un *creative brief*.
- Cómo seleccionar variantes ganadoras por microsegmento.
- Cómo construir una librería de prompts y borradores listos para activar.

## Outputs
- `nb18_genai_copy_briefs.csv`
- `nb18_genai_prompt_library.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

creatividades = tables["creatividades"].copy()
guardrails = tables["guardrails"].copy()
campanas = tables["campanas"].copy()

creatividades["performance_score"] = (
    creatividades["open_rate"].fillna(0) * 0.20
    + creatividades["ctr"].fillna(0) * 0.30
    + creatividades["cvr"].fillna(0) * 0.35
    + (creatividades["revenue_por_1000_envios_eur"].fillna(0) / creatividades["revenue_por_1000_envios_eur"].max()) * 0.15
)

best_variants = (
    creatividades.sort_values(["microsegmento", "canal", "performance_score"], ascending=[True, True, False])
                 .groupby(["microsegmento", "canal"])
                 .head(2)
                 .merge(campanas[["campaign_id", "objetivo", "copy_eje", "kpi_primario"]], on="campaign_id", how="left")
)

best_variants[[
    "creative_id", "campaign_id", "microsegmento", "canal", "categoria_objetivo",
    "tono", "beneficio_principal", "cta", "performance_score", "winning_variant_flag"
]].head(20)


,creative_id,campaign_id,microsegmento,canal,categoria_objetivo,tono,beneficio_principal,cta,performance_score,winning_variant_flag
0,CR-00061,CAM-2025-011,ACTIVO_BASE,EMAIL,multicategoría,práctico,ahorrar sin perder estilo,Activar oferta,0.166847,1
1,CR-00044,CAM-2025-005,ACTIVO_BASE,EMAIL,Iluminación,exclusivo,mejorar el ambiente del hogar,Activar oferta,0.165565,0
2,CR-00026,CAM-2024-009,ACTIVO_BASE,PUSH,multicategoría,práctico,ahorrar sin perder estilo,Reservar ahora,0.171175,1
3,CR-00054,CAM-2025-008,ACTIVO_BASE,PUSH,multicategoría,inspiracional,aprovechar una oferta útil,Descubrir,0.160276,1
4,CR-00016,CAM-2024-006,ALTO_TICKET_MUEBLES,EMAIL,Iluminación,exclusivo,completar tu compra,Activar oferta,0.171776,1
5,CR-00038,CAM-2025-003,ALTO_TICKET_MUEBLES,EMAIL,Muebles,exclusivo,elegir una colección premium,Descubrir,0.138209,0
6,CR-00008,CAM-2024-003,DECORACION_FRECUENTE,EMAIL,Decoración,urgencia_suave,sumar funcionalidad,Reservar ahora,0.146362,1
7,CR-00007,CAM-2024-003,DECORACION_FRECUENTE,EMAIL,Decoración,inspiracional,completar tu compra,Explorar colección,0.138486,0
8,CR-00071,CAM-2025-014,LOCAL_STORE,PUSH,Decoración,cercano,ver el producto en persona,Activar oferta,0.179087,1
9,CR-00072,CAM-2025-014,LOCAL_STORE,PUSH,Decoración,inspiracional,recoger hoy en tienda,Comprar hoy,0.169428,0


In [3]:

guardrail_text = " | ".join(guardrails.sort_values("severidad")["descripcion_regla"].tolist())

briefs = []
for _, row in best_variants.iterrows():
    draft_subject = row["asunto"]
    draft_body = (
        f"{row['copy_corto']} {row['prueba_social']}. "
        f"CTA sugerido: {row['cta']}. "
        f"Evitar infringir guardrails de marca."
    )
    prompt = f'''
Genera una pieza para FraSoHome.
Microsegmento: {row["microsegmento"]}.
Canal: {row["canal"]}.
Objetivo de campaña: {row["objetivo"]}.
Categoría objetivo: {row["categoria_objetivo"]}.
Tono: {row["tono"]}.
Beneficio principal: {row["beneficio_principal"]}.
CTA: {row["cta"]}.
Copy eje de campaña: {row["copy_eje"]}.
KPI primario: {row["kpi_primario"]}.
Guardrails obligatorios: {guardrail_text}

Devuelve:
1) asunto o titular,
2) copy corto,
3) copy largo,
4) justificación estratégica,
5) riesgo potencial y cómo mitigarlo.
'''.strip()

    briefs.append({
        "creative_id": row["creative_id"],
        "campaign_id": row["campaign_id"],
        "microsegmento": row["microsegmento"],
        "canal": row["canal"],
        "categoria_objetivo": row["categoria_objetivo"],
        "objetivo": row["objetivo"],
        "tono": row["tono"],
        "beneficio_principal": row["beneficio_principal"],
        "cta": row["cta"],
        "performance_score": row["performance_score"],
        "draft_subject": draft_subject,
        "draft_body": draft_body,
        "prompt_library_entry": prompt,
    })

copy_briefs = pd.DataFrame(briefs).sort_values(["microsegmento", "canal", "performance_score"], ascending=[True, True, False])
copy_briefs.head(20)


,creative_id,campaign_id,microsegmento,canal,categoria_objetivo,objetivo,tono,beneficio_principal,cta,performance_score,draft_subject,draft_body,prompt_library_entry
0,CR-00061,CAM-2025-011,ACTIVO_BASE,EMAIL,multicategoría,promocion,práctico,ahorrar sin perder estilo,Activar oferta,0.166847,Tu hogar: ideas que encajan contigo,Seleccionamos multicategoría con más probabili...,Genera una pieza para FraSoHome.\nMicrosegment...
1,CR-00044,CAM-2025-005,ACTIVO_BASE,EMAIL,Iluminación,cross_sell,exclusivo,mejorar el ambiente del hogar,Activar oferta,0.165565,Descubre una selección más precisa,Mejorar el ambiente del hogar con una propuest...,Genera una pieza para FraSoHome.\nMicrosegment...
2,CR-00026,CAM-2024-009,ACTIVO_BASE,PUSH,multicategoría,promocion,práctico,ahorrar sin perder estilo,Reservar ahora,0.171175,Tenemos novedades para tu casa,Seleccionamos multicategoría con más probabili...,Genera una pieza para FraSoHome.\nMicrosegment...
3,CR-00054,CAM-2025-008,ACTIVO_BASE,PUSH,multicategoría,reactivacion,inspiracional,aprovechar una oferta útil,Descubrir,0.160276,Tenemos novedades para tu casa,Aprovechar una oferta útil sin sobreexposición...,Genera una pieza para FraSoHome.\nMicrosegment...
4,CR-00016,CAM-2024-006,ALTO_TICKET_MUEBLES,EMAIL,Iluminación,cross_sell,exclusivo,completar tu compra,Activar oferta,0.171776,Descubre una selección más precisa,Completar tu compra sin sobreexposición promoc...,Genera una pieza para FraSoHome.\nMicrosegment...
5,CR-00038,CAM-2025-003,ALTO_TICKET_MUEBLES,EMAIL,Muebles,upsell,exclusivo,elegir una colección premium,Descubrir,0.138209,Descubre una selección más precisa,Elegir una colección premium con una propuesta...,Genera una pieza para FraSoHome.\nMicrosegment...
6,CR-00008,CAM-2024-003,DECORACION_FRECUENTE,EMAIL,Decoración,cross_sell,urgencia_suave,sumar funcionalidad,Reservar ahora,0.146362,Solo para ti: Primavera Decoración,Sumar funcionalidad con una propuesta de decor...,Genera una pieza para FraSoHome.\nMicrosegment...
7,CR-00007,CAM-2024-003,DECORACION_FRECUENTE,EMAIL,Decoración,cross_sell,inspiracional,completar tu compra,Explorar colección,0.138486,Descubre una selección más precisa,Seleccionamos decoración con más probabilidad ...,Genera una pieza para FraSoHome.\nMicrosegment...
8,CR-00071,CAM-2025-014,LOCAL_STORE,PUSH,Decoración,trafico_tienda,cercano,ver el producto en persona,Activar oferta,0.179087,Tenemos novedades para tu casa,Ver el producto en persona sin sobreexposición...,Genera una pieza para FraSoHome.\nMicrosegment...
9,CR-00072,CAM-2025-014,LOCAL_STORE,PUSH,Decoración,trafico_tienda,inspiracional,recoger hoy en tienda,Comprar hoy,0.169428,Comprar hoy tu selección de hoy,Recoger hoy en tienda sin sobreexposición prom...,Genera una pieza para FraSoHome.\nMicrosegment...


In [4]:

prompt_library = copy_briefs[[
    "creative_id", "campaign_id", "microsegmento", "canal", "categoria_objetivo",
    "objetivo", "performance_score", "prompt_library_entry"
]].copy()

copy_briefs["recomendacion_uso"] = np.where(
    copy_briefs["performance_score"] >= copy_briefs["performance_score"].quantile(0.75),
    "Usar como base de nuevas variantes y test A/B",
    "Usar como referencia secundaria o control"
)

copy_briefs[["microsegmento", "canal", "draft_subject", "draft_body", "recomendacion_uso"]].head(15)


,microsegmento,canal,draft_subject,draft_body,recomendacion_uso
0,ACTIVO_BASE,EMAIL,Tu hogar: ideas que encajan contigo,Seleccionamos multicategoría con más probabili...,Usar como referencia secundaria o control
1,ACTIVO_BASE,EMAIL,Descubre una selección más precisa,Mejorar el ambiente del hogar con una propuest...,Usar como referencia secundaria o control
2,ACTIVO_BASE,PUSH,Tenemos novedades para tu casa,Seleccionamos multicategoría con más probabili...,Usar como referencia secundaria o control
3,ACTIVO_BASE,PUSH,Tenemos novedades para tu casa,Aprovechar una oferta útil sin sobreexposición...,Usar como referencia secundaria o control
4,ALTO_TICKET_MUEBLES,EMAIL,Descubre una selección más precisa,Completar tu compra sin sobreexposición promoc...,Usar como referencia secundaria o control
5,ALTO_TICKET_MUEBLES,EMAIL,Descubre una selección más precisa,Elegir una colección premium con una propuesta...,Usar como referencia secundaria o control
6,DECORACION_FRECUENTE,EMAIL,Solo para ti: Primavera Decoración,Sumar funcionalidad con una propuesta de decor...,Usar como referencia secundaria o control
7,DECORACION_FRECUENTE,EMAIL,Descubre una selección más precisa,Seleccionamos decoración con más probabilidad ...,Usar como referencia secundaria o control
8,LOCAL_STORE,PUSH,Tenemos novedades para tu casa,Ver el producto en persona sin sobreexposición...,Usar como base de nuevas variantes y test A/B
9,LOCAL_STORE,PUSH,Comprar hoy tu selección de hoy,Recoger hoy en tienda sin sobreexposición prom...,Usar como referencia secundaria o control



## Lectura de negocio

La GenAI debe entrar después de tres filtros:

1. **segmento correcto**,
2. **aprendizaje histórico de desempeño**,
3. **guardrails de marca y legales**.

Así se evita usar IA generativa como “máquina de copy” sin contexto y se convierte en una herramienta de personalización controlada.


In [5]:

path_briefs = mpu.export_csv(copy_briefs, "nb18_genai_copy_briefs.csv")
path_prompts = mpu.export_csv(prompt_library, "nb18_genai_prompt_library.csv")

print("Guardado:", path_briefs)
print("Guardado:", path_prompts)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb18_genai_copy_briefs.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb18_genai_prompt_library.csv
